# 03 - EDA de negocio

Objetivo: entender el comportamiento comercial antes de modelar metricas avanzadas: historico, ventas, margen, devoluciones y concentracion de valor.

In [ ]:
from pathlib import Path
import os

import pandas as pd
from sqlalchemy import create_engine, text

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DB_NAME = os.getenv("DB_NAME", "saleshealth")
DB_USER = os.getenv("DB_USER", os.getenv("USER", "postgres"))
DB_HOST = os.getenv("DB_HOST", "")
DB_PORT = os.getenv("DB_PORT", "")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")

if DB_HOST:
    auth = f"{DB_USER}:{DB_PASSWORD}@" if DB_PASSWORD else f"{DB_USER}@"
    port = f":{DB_PORT}" if DB_PORT else ""
    DATABASE_URL = os.getenv("DATABASE_URL", f"postgresql+psycopg2://{auth}{DB_HOST}{port}/{DB_NAME}")
else:
    DATABASE_URL = os.getenv("DATABASE_URL", f"postgresql+psycopg2:///{DB_NAME}")

engine = create_engine(DATABASE_URL)

def q(sql):
    return pd.read_sql_query(text(sql), engine)

In [ ]:
ventas_mes = q('''
select date_trunc('month', s.sale_date)::date as mes,
       count(distinct s.sale_id) as ventas,
       count(si.sale_item_id) as lineas,
       round(sum(si.subtotal)::numeric, 2) as ingresos
from sale s
join sale_item si on si.sale_id = s.sale_id
group by 1
order by 1;
''')
ventas_mes.tail()

In [ ]:
ax = ventas_mes.plot(x="mes", y="ingresos", figsize=(10, 4), title="Ingresos mensuales")
ax.set_xlabel("Mes")
ax.set_ylabel("Ingresos")

In [ ]:
top_productos = q('''
select p.product_id, p.name as producto,
       count(*) as lineas,
       sum(si.quantity) as unidades,
       round(sum(si.subtotal)::numeric, 2) as ingresos
from sale_item si
join product p on p.product_id = si.product_id
group by p.product_id, p.name
order by ingresos desc
limit 10;
''')
top_productos

In [ ]:
devoluciones = q('''
select rr.reason as motivo,
       count(*) as devoluciones,
       sum(ri.quantity) as unidades_devueltas
from return_item ri
left join return_reason rr on rr.reason_id = ri.reason_id
group by rr.reason
order by devoluciones desc;
''')
devoluciones

In [ ]:
clientes = q('''
select s.customer_id,
       round(sum(si.subtotal)::numeric, 2) as ingresos
from sale s
join sale_item si on si.sale_id = s.sale_id
group by s.customer_id
order by ingresos desc;
''')
clientes["ingresos_acumulados_pct"] = clientes["ingresos"].cumsum() / clientes["ingresos"].sum()
clientes["clientes_acumulados_pct"] = (clientes.index + 1) / len(clientes)
clientes.head(10)

In [ ]:
ax = clientes.plot(x="clientes_acumulados_pct", y="ingresos_acumulados_pct", figsize=(6, 4), title="Concentracion de ingresos")
ax.set_xlabel("% clientes acumulado")
ax.set_ylabel("% ingresos acumulado")